In [1]:
!pip install tqdm
!pip install transformers==4.40.1
!pip install sentencepiece
!pip install datasets
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl
!pip install triton
!pip install bitsandbytes
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install xformers
!pip install pytorch-cuda==12.1 torch xformers
#!pip install --no-deps xformers trl peft accelerate bitsandbytes
#!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install hyperopt
!pip install optuna

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-kh370ktt/unsloth_46727aa8df3243df81346b6f10147acb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-kh370ktt/unsloth_46727aa8df3243df81346b6f10147acb
  Resolved https://github.com/unslothai/unsloth.git to commit 556f396b3cd63da525e2cdac1a5f1c606f4494ce
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.16.4-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
Using cached huggingface_hub-1.16.4-py3-none-any.whl (668 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting

In [2]:
!python -m xformers.info
!python -m bitsandbytes
!nvidia-smi


xFormers 0.0.35
memory_efficient_attention.ckF:                    unavailable
memory_efficient_attention.ckB:                    unavailable
memory_efficient_attention.ck_splitKF:             unavailable
memory_efficient_attention.cutlassF-pt:            available
memory_efficient_attention.cutlassB-pt:            available
memory_efficient_attention.cutlassF-blackwell:     unavailable
memory_efficient_attention.cutlassB-blackwell:     unavailable
memory_efficient_attention.fa2F@2.5.7-pt:          available
memory_efficient_attention.fa2B@2.5.7-pt:          available
memory_efficient_attention.fa3F@0.0.0:             unavailable
memory_efficient_attention.fa3B@0.0.0:             unavailable
memory_efficient_attention.fa3F_splitKV@0.0.0:     unavailable
memory_efficient_attention.triton_splitKF:         available
indexing.scaled_index_addF:                        unavailable
indexing.scaled_index_addB:                        unavailable
indexing.index_select:                           

In [3]:
import json
import torch
from datasets import load_dataset
from huggingface_hub import notebook_login
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel
print(torch.__version__)
print(torch.version.cuda)

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
2.11.0+cu128
12.8


In [4]:
# Defining the configuration for the base model, LoRA and training
config = {
    "hugging_face_username":"aravag",
    "model_config": {
        "base_model":"mistralai/Mistral-7B-Instruct-v0.2", # The base model
        "finetuned_model":"aravag/Mercury-STD-Mistral", # The finetuned model
        "max_seq_length": 2048, # The maximum sequence length
       # "dtype":torch.float16, # The data type
       #  "dtype": torch.float32,  # Use float32 instead of half CUDA capability < 8
          "dtype" : None, # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+

        "load_in_4bit": True, # Load the model in 4-bit
    },
    "lora_config": {
      "r": 16, # The number of LoRA layers 8, 16, 32, 64
      "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # The target modules
      "lora_alpha":16, # The alpha value for LoRA
      #"lora_alpha":15, # The alpha value for LoRA by search grid
      "lora_dropout":0, # The dropout value for LoRA
      "bias":"none", # The bias for LoRA
      "use_gradient_checkpointing":True, # Use gradient checkpointing
      "use_rslora":False, # Use RSLora
      "use_dora":False, # Use DoRa
      "loftq_config":None # The LoFTQ configuration
    },

    "training_config": {
        "per_device_train_batch_size": 2, # The batch size
        #"per_device_train_batch_size": 6, # The batch size by search grid
        "gradient_accumulation_steps": 4, # The gradient accumulation steps
        #"gradient_accumulation_steps": 7, # The gradient accumulation steps by search grid
        "warmup_steps": 5, # The warmup steps
        "max_steps":0, # The maximum steps (0 if the epochs are defined)
        "num_train_epochs": 1, # The number of training epochs(0 if the maximum steps are defined)
        "learning_rate": 2e-4, # The learning rate
        #"learning_rate": 9.5e-05, # The learning rate  by search grid
        "fp16": not torch.cuda.is_bf16_supported(), # The fp16
        "bf16": torch.cuda.is_bf16_supported(), # The bf16
        "logging_steps": 1, # The logging steps
        "optim" :"adamw_8bit", # The optimizer
        "weight_decay" : 0.01,  # The weight decay
        "lr_scheduler_type": "linear", # The learning rate scheduler
        "seed" : 42, # The seed
        "output_dir" : "outputs", # The output directory
    }
}

In [5]:
config_dataset={    "training_dataset": {
        "name": "std_conversations", # The dataset name
        "split": "train",  # The dataset split
        "input_fields": ["question", "context"] ,# The input fields
        "input_field": "text",# The input field
    },
                }

In [6]:
# Loading the model and the tokinizer for the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = config.get("model_config").get("base_model"),
    max_seq_length = config.get("model_config").get("max_seq_length"),
    dtype = config.get("model_config").get("dtype"),
    load_in_4bit = config.get("model_config").get("load_in_4bit"),

)

==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.13k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

In [7]:
# Setup for QLoRA/LoRA peft of the base model
model = FastLanguageModel.get_peft_model(
    model,
    r = config.get("lora_config").get("r"),
    target_modules = config.get("lora_config").get("target_modules"),
    lora_alpha = config.get("lora_config").get("lora_alpha"),
    lora_dropout = config.get("lora_config").get("lora_dropout"),
    bias = config.get("lora_config").get("bias"),
    use_gradient_checkpointing = config.get("lora_config").get("use_gradient_checkpointing"),
    random_state = 42,
    use_rslora = config.get("lora_config").get("use_rslora"),
    use_dora = config.get("lora_config").get("use_dora"),
    loftq_config = config.get("lora_config").get("loftq_config"),
)


Unsloth 2026.5.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [8]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.get("model_config").get("base_model"))


tokenizer.add_eos_token = True
tokenizer.pad_token_id = 0
tokenizer.padding_side = "left"

# # Loading the training dataset
# train_dataset = load_dataset(config_dataset.get("training_dataset").get("name"), split = config_dataset.get("training_dataset").get("split"))

import pandas as pd
from datasets import Dataset

df = pd.read_csv('std_conversations.csv')
df = df.rename(columns={"Description": "question", "Doctor": "context"})
train_dataset = Dataset.from_pandas(df)


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [9]:
# Select the first 100 rows of the dataset
test_dataset = train_dataset.select(range(100))

In [10]:
medical_prompt = """You are an AI Medical Assistant Chatbot, trained to answer medical questions. Below is an instruction that describes a task, paired with an response context. Write a response that appropriately completes the request.

### Instruction:
{}


### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["question"]
    outputs      = examples["context"]
    texts = []
    for instruction, output in zip(instructions,  outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = medical_prompt.format(instruction,  output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

test_dataset= test_dataset.map(formatting_prompts_func, batched = True,)



test_dataset['text'][1]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

'You are an AI Medical Assistant Chatbot, trained to answer medical questions. Below is an instruction that describes a task, paired with an response context. Write a response that appropriately completes the request.\n\n### Instruction:\nQ. Are my symptoms due to HIV infection? I had a high-risk exposure 15 months ago.\n\n\n### Response:\nHi. The test kits used to diagnose HIV are highly sensitive and specific and gives accurate results. The majority of the people who are infected with HIV (human immunodeficiency virus) develop antibodies by three months, and even in the rare instances of late seroconversions, they develop HIV antibodies by six months. You can rely on your HIV screening test results. If there had been no other exposure other than that, you do not require any further test. The symptoms which you mentioned could be due to various reasons and cannot be attributed to HIV. In fact, there are no specific symptoms or signs which can lead to the diagnosis of HIV. As per the a

In [11]:
test_dataset

Dataset({
    features: ['question', 'context', 'text'],
    num_rows: 100
})

In [15]:
from trl import SFTTrainer, SFTConfig

trainer_test = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = test_dataset,
    args = SFTConfig(
        max_seq_length = config.get("model_config").get("max_seq_length"),
        dataset_text_field = config_dataset.get("training_dataset").get("input_field"),
        dataset_num_proc = 2,
        packing = False,
        per_device_train_batch_size = config.get("training_config").get("per_device_train_batch_size"),
        gradient_accumulation_steps = config.get("training_config").get("gradient_accumulation_steps"),
        warmup_steps = config.get("training_config").get("warmup_steps"),
        max_steps = config.get("training_config").get("max_steps"),
        num_train_epochs = config.get("training_config").get("num_train_epochs"),
        learning_rate = config.get("training_config").get("learning_rate"),
        fp16 = config.get("training_config").get("fp16"),
        bf16 = config.get("training_config").get("bf16"),
        logging_steps = config.get("training_config").get("logging_steps"),
        optim = config.get("training_config").get("optim"),
        weight_decay = config.get("training_config").get("weight_decay"),
        lr_scheduler_type = config.get("training_config").get("lr_scheduler_type"),
        seed = 42,
        output_dir = config.get("training_config").get("output_dir"),
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


## Method 1 optuna

In [16]:
from optuna import create_study, Trial

# Define search space
search_space = {
  "learning_rate": [1e-5, 5e-5, 1e-4, 2e-4],
  "per_device_train_batch_size": [2, 4, 8],
  "lora_alpha": [8, 16, 32],
}

def objective(trial):
  # Set hyperparameters based on trial values
  config["training_config"]["learning_rate"] = trial.suggest_float("learning_rate", search_space["learning_rate"][0], search_space["learning_rate"][-1])
  config["training_config"]["per_device_train_batch_size"] = trial.suggest_int("per_device_train_batch_size", search_space["per_device_train_batch_size"][0], search_space["per_device_train_batch_size"][-1])
  config["lora_config"]["lora_alpha"] = trial.suggest_int("lora_alpha", search_space["lora_alpha"][0], search_space["lora_alpha"][-1])

  # Train the model with the current hyperparameters
  try:
      trainer_stats = trainer_test.train()  # Assuming this trains the model
      return trainer_stats["train_loss"]  # Assuming this is the metric to minimize
  except Exception as e:
      return float("inf")  # Assign a high value if training fails

study = create_study(direction="minimize")
study.optimize(objective, n_trials=2)  # Adjust the number of trials

# Access the best trial and its hyperparameters after optimization
best_trial = study.best_trial
best_params = best_trial.params

print("Best Trial:", best_trial.number)
print("Best Hyperparameters:", best_params)
print("Best Training Loss:", best_trial.value)


[I 2026-05-27 10:24:17,827] A new study created in memory with name: no-name-d8d55a0b-f347-4b56-9402-e305e31abcb3
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 1 | Total steps = 0
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.519675


[I 2026-05-27 10:24:47,269] Trial 0 finished with value: inf and parameters: {'learning_rate': 3.0178113291803308e-05, 'per_device_train_batch_size': 3, 'lora_alpha': 14}. Best is trial 0 with value: inf.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 1 | Total steps = 0
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss
1,3.519675


[I 2026-05-27 10:25:00,451] Trial 1 finished with value: inf and parameters: {'learning_rate': 8.14298320235685e-05, 'per_device_train_batch_size': 6, 'lora_alpha': 21}. Best is trial 0 with value: inf.


Best Trial: 0
Best Hyperparameters: {'learning_rate': 3.0178113291803308e-05, 'per_device_train_batch_size': 3, 'lora_alpha': 14}
Best Training Loss: inf


## Analyzing Hyperparameters:

*  **Batch Size**: Generally, increasing the batch size can improve


training speed by utilizing hardware resources more efficiently. However, there's a limit beyond which performance degrades. You can tune the batch size within a reasonable range (e.g., 2, 4, 8, 16) to see its impact.
* **Learning Rate**: A higher learning rate can accelerate training initially. But, a too high value can lead to unstable training and potentially slower convergence. Consider a range of learning rates (e.g., log-uniform distribution between 1e-5 and 1e-3) for exploration.
* **Gradient Accumulation Steps**: This technique accumulates gradients over multiple batches before updating model weights. It can help reduce memory requirements but might slow down training per epoch. Experiment with different accumulation steps (e.g., 1, 2, 4) to find a balance.
* **Optimizer Choice**: Some optimizers like Adam or SGD with momentum can be faster than others depending on the model and dataset. Explore different optimizers and their hyperparameters (e.g., momentum coefficient) to see if they lead to faster convergence.
## Additional Considerations:

Early Stopping: Implement early stopping to automatically terminate training if the validation loss doesn't improve for a certain number of epochs. This can save training time if the model starts overfitting.
Warmup Steps: A gradual increase in the learning rate during the initial training phase (warmup steps) can improve stability and potentially accelerate convergence compared to a fixed learning rate from the beginning.


* Experimentation and Profiling:

The best hyperparameters for faster training depend on your specific model, dataset, and hardware. You'll need to experiment with different configurations using tools like Hyperopt to find the optimal settings.
Consider using profiling tools to identify bottlenecks in your training pipeline. This can help you focus on optimizing specific parts of the training process that are most time-consuming.
By analyzing these hyperparameters and implementing techniques like early stopping and warmup steps, you can potentially achieve faster fine-tuning while maintaining good model performance.

In [ ]:
## Method 1b Speed

In [17]:
from optuna import create_study, Trial
import time  # Assuming you can use time.time() to measure training time

# Define search space with additional hyperparameter
search_space = {
  "learning_rate": [1e-5, 5e-5, 1e-4, 2e-4],
  "per_device_train_batch_size": [2, 4, 8],
  "lora_alpha": [8, 16, 32],
  "gradient_accumulation_steps": [1, 2, 4, 8],  # Added gradient accumulation steps
}

def objective(trial):
  # Set hyperparameters based on trial values
  config["training_config"]["learning_rate"] = trial.suggest_float("learning_rate", search_space["learning_rate"][0], search_space["learning_rate"][-1])
  config["training_config"]["per_device_train_batch_size"] = trial.suggest_int("per_device_train_batch_size", search_space["per_device_train_batch_size"][0], search_space["per_device_train_batch_size"][-1])
  config["training_config"]["gradient_accumulation_steps"] = trial.suggest_int("gradient_accumulation_steps", search_space["gradient_accumulation_steps"][0], search_space["gradient_accumulation_steps"][-1])
  config["lora_config"]["lora_alpha"] = trial.suggest_int("lora_alpha", search_space["lora_alpha"][0], search_space["lora_alpha"][-1])

  # Train the model with the current hyperparameters
  start_time = time.time()
  try:
      trainer_stats = trainer_test.train()
      training_time = time.time() - start_time
      return training_time  # Minimize training time
  except Exception as e:
      return float("inf")  # Assign a high value if training fails

study = create_study(direction="minimize")
study.optimize(objective, n_trials=2)  # Adjust the number of trials

# Access the best trial and its hyperparameters after optimization
best_trial = study.best_trial
best_params = best_trial.params

print("Best Trial:", best_trial.number)
print("Best Hyperparameters (Likely Fastest):", best_params)
print("Best Training Time:", best_trial.value, "seconds")

[I 2026-05-27 10:25:00,498] A new study created in memory with name: no-name-067df9cd-4eda-4b42-87ad-43a7d5276c1a
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 1 | Total steps = 0
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss
1,3.519675


[I 2026-05-27 10:25:17,640] Trial 0 finished with value: 17.132619857788086 and parameters: {'learning_rate': 2.7037269966786433e-05, 'per_device_train_batch_size': 6, 'gradient_accumulation_steps': 5, 'lora_alpha': 9}. Best is trial 0 with value: 17.132619857788086.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 1 | Total steps = 0
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss
1,3.519675


[I 2026-05-27 10:25:31,735] Trial 1 finished with value: 14.08804440498352 and parameters: {'learning_rate': 0.00011416436527434034, 'per_device_train_batch_size': 7, 'gradient_accumulation_steps': 6, 'lora_alpha': 32}. Best is trial 1 with value: 14.08804440498352.


Best Trial: 1
Best Hyperparameters (Likely Fastest): {'learning_rate': 0.00011416436527434034, 'per_device_train_batch_size': 7, 'gradient_accumulation_steps': 6, 'lora_alpha': 32}
Best Training Time: 14.08804440498352 seconds


In [18]:
import hyperopt
from hyperopt import hp
from hyperopt import Trials
from hyperopt import fmin, tpe, Trials
# Define the search space for hyperparameters
space = {
  'learning_rate': hp.loguniform('learning_rate', -5, -1),  # Learning rate in log scale
  'lora_alpha': hp.quniform('lora_alpha', 1, 32, 1),  # LoRA alpha with quantized steps
  'lora_dropout': hp.uniform('lora_dropout', 0, 0.5),  # LoRA dropout rate
  # Uncomment these if you want to tune them
  # 'per_device_train_batch_size': hp.quniform('per_device_train_batch_size', 2, 16, 1),
  # 'gradient_accumulation_steps': hp.quniform('gradient_accumulation_steps', 1, 8, 1),
  # 'warmup_steps': hp.quniform('warmup_steps', 0, 1000, 1),
  # 'num_train_epochs': hp.quniform('num_train_epochs', 1, 5, 1),
}
def objective(params):
  # Set hyperparameters in the config dictionary (assuming it's defined elsewhere)
  config['training_config']['learning_rate'] = params['learning_rate']
  config['lora_config']['lora_alpha'] = params['lora_alpha']
  config['lora_config']['lora_dropout'] = params['lora_dropout']
  # ... Set other hyperparameters from params dictionary ...
  #config['training_config']['per_device_train_batch_size'] = params['per_device_train_batch_size']
  #config['training_config']['gradient_accumulation_steps'] = params['gradient_accumulation_steps']
  #config['training_config']['warmup_steps'] = params['warmup_steps']
  #config['training_config']['num_train_epochs'] = params['num_train_epochs']

  # Load the model and tokenizer (assuming these are defined elsewhere)
  try:
      model, tokenizer = FastLanguageModel.from_pretrained(
          model_name=config.get("model_config").get("base_model"),
          max_seq_length=config.get("model_config").get("max_seq_length"),
          dtype=config.get("model_config").get("dtype"),
          load_in_4bit=config.get("model_config").get("load_in_4bit"),
      )
  except Exception as e:
      print(f"Error loading model and tokenizer: {e}")
      return float("inf")  # Return high value for errors

  # Setup LoRA for the model (assuming FastLanguageModel supports LoRA)
  try:
      model = FastLanguageModel.get_peft_model(
          model,
          r=config.get("lora_config").get("r"),
          target_modules=config.get("lora_config").get("target_modules"),
          lora_alpha=params['lora_alpha'],
          lora_dropout=params['lora_dropout'],
          bias=config.get("lora_config").get("bias"),
          use_gradient_checkpointing=config.get("lora_config").get("use_gradient_checkpointing"),
          random_state=42,
          use_rslora=config.get("lora_config").get("use_rslora"),
          use_dora=config.get("lora_config").get("use_dora"),
          loftq_config=config.get("lora_config").get("loftq_config")
      )
  except Exception as e:
      print(f"Error setting up LoRA: {e}")
      return float("inf")  # Return high value for errors
  # Train the model on the test dataset (assuming SFTTrainer and training arguments are defined)
  try:
      trainer = SFTTrainer(
          model=model,
          tokenizer=tokenizer,
          train_dataset=test_dataset,
          dataset_text_field=config_dataset.get("training_dataset").get("input_field"),
          max_seq_length=config.get("model_config").get("max_seq_length"),
          dataset_num_proc=2,
          packing=False,
          args=TrainingArguments(
              per_device_train_batch_size=int(params['per_device_train_batch_size']),
              gradient_accumulation_steps=params['gradient_accumulation_steps'],
              warmup_steps=params['warmup_steps'],
              max_steps=config.get("training_config").get("max_steps"),
              num_train_epochs=params['num_train_epochs'],
              learning_rate=params['learning_rate'],
              fp16=config.get("training_config").get("fp16"),
              bf16=config.get("training_config").get("bf16"),
              logging_steps=config.get("training_config").get("logging_steps"),
              optim=config.get("training_config").get("optim"),
              weight_decay=config.get("training_config").get("weight_decay"),
              lr_scheduler_type=config.get("training_config").get("lr_scheduler_type"),
              seed=42,
              output_dir=config.get("training_config").get("output_dir")
          )
      )
      trainer_stats = trainer.train()
      return trainer_stats.loss  # Assuming loss is the metric to minimize
  except Exception as e:
      print(f"Error during training: {e}")
      return float("inf")  # Return high value for failed trials

# Create a Trials object to track hyperparameter evaluations
trials = Trials()

# Run hyperparameter optimization using TPE algorithm
best = fmin(objective, space, algo=tpe.suggest, trials=trials, max_evals=2)

# Print the best hyperparameters found during optimization
print("Best Hyperparameters:", best)


==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
  0%|          | 0/2 [00:00<?, ?trial/s, best loss=?]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.13089840078441645.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.8 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Error during training: 'per_device_train_batch_size'
==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
 50%|█████     | 1/2 [00:25<00:25, 25.31s/trial, best loss: inf]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.09608198982117772.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Error during training: 'per_device_train_batch_size'
100%|██████████| 2/2 [00:50<00:00, 25.42s/trial, best loss: inf]
Best Hyperparameters: {'learning_rate': np.float64(0.1999520983329925), 'lora_alpha': np.float64(11.0), 'lora_dropout': np.float64(0.13089840078441645)}


In [19]:
import hyperopt
from hyperopt import hp
from hyperopt import Trials
from hyperopt import fmin, tpe, Trials

# Define the search space for hyperparameters with uncommented additions
space = {
  'learning_rate': hp.loguniform('learning_rate', -5, -1),  # Learning rate in log scale
  'lora_alpha': hp.quniform('lora_alpha', 1, 32, 1),  # LoRA alpha with quantized steps
  'lora_dropout': hp.uniform('lora_dropout', 0, 0.5),  # LoRA dropout rate
  'per_device_train_batch_size': hp.quniform('per_device_train_batch_size', 2, 16, 1),  # Added for exploration
  'gradient_accumulation_steps': hp.quniform('gradient_accumulation_steps', 1, 8, 1),  # Added for exploration
  # Uncomment these if you want to tune other hyperparameters
  # 'warmup_steps': hp.quniform('warmup_steps', 0, 1000, 1),
  # 'num_train_epochs': hp.quniform('num_train_epochs', 1, 5, 1),
}


def objective(params):
  # Set hyperparameters in the config dictionary (assuming it's defined elsewhere)
  config['training_config']['learning_rate'] = params['learning_rate']
  config['lora_config']['lora_alpha'] = params['lora_alpha']
  config['lora_config']['lora_dropout'] = params['lora_dropout']
  config['training_config']['per_device_train_batch_size'] = params['per_device_train_batch_size']
  config['training_config']['gradient_accumulation_steps'] = params['gradient_accumulation_steps']
  # ... Set other hyperparameters from params dictionary ...

  # Load the model and tokenizer (assuming these are defined elsewhere)
  try:
      model, tokenizer = FastLanguageModel.from_pretrained(
          model_name=config.get("model_config").get("base_model"),
          max_seq_length=config.get("model_config").get("max_seq_length"),
          dtype=config.get("model_config").get("dtype"),
          load_in_4bit=config.get("model_config").get("load_in_4bit"),
      )
  except Exception as e:
      print(f"Error loading model and tokenizer: {e}")
      return float("inf")  # Return high value for errors

  # Setup LoRA for the model (assuming FastLanguageModel supports LoRA)
  try:
      model = FastLanguageModel.get_peft_model(
          model,
          r=config.get("lora_config").get("r"),
          target_modules=config.get("lora_config").get("target_modules"),
          lora_alpha=params['lora_alpha'],
          lora_dropout=params['lora_dropout'],
          bias=config.get("lora_config").get("bias"),
          use_gradient_checkpointing=config.get("lora_config").get("use_gradient_checkpointing"),
          random_state=42,
          use_rslora=config.get("lora_config").get("use_rslora"),
          use_dora=config.get("lora_config").get("use_dora"),
          loftq_config=config.get("lora_config").get("loftq_config")
      )
  except Exception as e:
      print(f"Error setting up LoRA: {e}")
      return float("inf")  # Return high value for errors

  # Train the model on the test dataset (assuming SFTTrainer and training arguments are defined)
  try:
      trainer = SFTTrainer(
          model=model,
          tokenizer=tokenizer,
          train_dataset=test_dataset,
          dataset_text_field=config_dataset.get("training_dataset").get("input_field"),
          max_seq_length=config.get("model_config").get("max_seq_length"),
          dataset_num_proc=2,
          packing=False,
          args=TrainingArguments(
              per_device_train_batch_size=int(params['per_device_train_batch_size']),
              gradient_accumulation_steps=params['gradient_accumulation_steps'],
              warmup_steps=params['warmup_steps'],
              max_steps=config.get("training_config").get("max_steps"),
              num_train_epochs=params['num_train_epochs'],
              learning_rate=params['learning_rate'],
              fp16=config.get("training_config").get("fp16"),
              bf16=config.get("training_config").get("bf16"),
              logging_steps=config.get("training_config").get("logging_steps"),
              optim=config.get("training_config").get("optim"),
              weight_decay=config.get("training_config").get("weight_decay"),
              lr_scheduler_type=config.get("training_config").get("lr_scheduler_type"),
              seed=42,
              output_dir=config.get("training_config").get("output_dir")
          )
      )
      trainer_stats = trainer.train()
      return trainer_stats.loss  # Assuming loss is the metric to minimize
  except Exception as e:
      print(f"Error during training: {e}")
      return float("inf")  # Return high value for failed trials

# Create a Trials object to track hyperparameter evaluations
trials = Trials()

# Run hyperparameter optimization using TPE algorithm
best = fmin(objective, space, algo=tpe.suggest, trials=trials, max_evals=2)

# Print the best hyperparameters found during optimization
print("Best Hyperparameters:", best)


==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
  0%|          | 0/2 [00:00<?, ?trial/s, best loss=?]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.35670944376102126.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Error during training: 'warmup_steps'
==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
 50%|█████     | 1/2 [00:25<00:24, 24.89s/trial, best loss: inf]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.23440908509920583.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Error during training: 'warmup_steps'
100%|██████████| 2/2 [00:45<00:00, 22.74s/trial, best loss: inf]
Best Hyperparameters: {'gradient_accumulation_steps': np.float64(2.0), 'learning_rate': np.float64(0.06898975550603895), 'lora_alpha': np.float64(25.0), 'lora_dropout': np.float64(0.35670944376102126), 'per_device_train_batch_size': np.float64(13.0)}


In [ ]:
## Method

In [20]:
import hyperopt
from hyperopt import hp
from hyperopt import Trials
from hyperopt import fmin, tpe, Trials
import time  # Import time for measuring training duration

# Define the search space for hyperparameters with uncommented additions
space = {
  'learning_rate': hp.loguniform('learning_rate', -5, -1),  # Learning rate in log scale
  'lora_alpha': hp.quniform('lora_alpha', 1, 32, 1),  # LoRA alpha with quantized steps
  'lora_dropout': hp.uniform('lora_dropout', 0, 0.5),  # LoRA dropout rate
  'per_device_train_batch_size': hp.quniform('per_device_train_batch_size', 2, 16, 1),  # Added for exploration
  'gradient_accumulation_steps': hp.quniform('gradient_accumulation_steps', 1, 8, 1),  # Added for exploration
  # Uncomment these if you want to tune other hyperparameters
  # 'warmup_steps': hp.quniform('warmup_steps', 0, 1000, 1),
  # 'num_train_epochs': hp.quniform('num_train_epochs', 1, 5, 1),
}


def objective(params):
  # Set hyperparameters in the config dictionary (assuming it's defined elsewhere)
  config['training_config']['learning_rate'] = params['learning_rate']
  config['lora_config']['lora_alpha'] = params['lora_alpha']
  config['lora_config']['lora_dropout'] = params['lora_dropout']
  config['training_config']['per_device_train_batch_size'] = params['per_device_train_batch_size']
  config['training_config']['gradient_accumulation_steps'] = params['gradient_accumulation_steps']
  # ... Set other hyperparameters from params dictionary ...

  # Load the model and tokenizer (assuming these are defined elsewhere)
  try:
      model, tokenizer = FastLanguageModel.from_pretrained(
          model_name=config.get("model_config").get("base_model"),
          max_seq_length=config.get("model_config").get("max_seq_length"),
          dtype=config.get("model_config").get("dtype"),
          load_in_4bit=config.get("model_config").get("load_in_4bit"),
      )
  except Exception as e:
      print(f"Error loading model and tokenizer: {e}")
      return float("inf")  # Return high value for errors

  # Setup LoRA for the model (assuming FastLanguageModel supports LoRA)
  try:
      model = FastLanguageModel.get_peft_model(
          model,
          r=config.get("lora_config").get("r"),
          target_modules=config.get("lora_config").get("target_modules"),
          lora_alpha=params['lora_alpha'],
          lora_dropout=params['lora_dropout'],
          bias=config.get("lora_config").get("bias"),
          use_gradient_checkpointing=config.get("lora_config").get("use_gradient_checkpointing"),
          random_state=42,
          use_rslora=config.get("lora_config").get("use_rslora"),
          use_dora=config.get("lora_config").get("use_dora"),
          loftq_config=config.get("lora_config").get("loftq_config")
      )
  except Exception as e:
      print(f"Error setting up LoRA: {e}")
      return float("inf")  # Return high value for errors

  # Train the model on the test dataset (assuming SFTTrainer and training arguments are defined)
  try:
      start_time = time.time()  # Measure training start time
      trainer = SFTTrainer(
          model=model,
          tokenizer=tokenizer,
          train_dataset=test_dataset,
          dataset_text_field=config_dataset.get("training_dataset").get("input_field"),
          max_seq_length=config.get("model_config").get("max_seq_length"),
          dataset_num_proc=2,
          packing=False,
          args=TrainingArguments(
              per_device_train_batch_size=int(params['per_device_train_batch_size']),
              gradient_accumulation_steps=params['gradient_accumulation_steps'],
              warmup_steps=params['warmup_steps'],
              max_steps=config.get("training_config").get("max_steps"),
              num_train_epochs=params['num_train_epochs'],
              learning_rate=params['learning_rate'],
              fp16=config.get("training_config").get("fp16"),
              bf16=config.get("training_config").get("bf16"),
              logging_steps=config.get("training_config").get("logging_steps"),
              optim=config.get("training_config").get("optim"),
              weight_decay=config.get("training_config").get("weight_decay"),
              lr_scheduler_type=config.get("training_config").get("lr_scheduler_type"),
              seed=42,
              output_dir=config.get("training_config").get("output_dir")
          )
      )
      trainer_stats = trainer.train()
      end_time = time.time()  # Measure training end time
      training_time = end_time - start_time  # Calculate training time

      return training_time  # Return training time for minimization
  except Exception as e:
      print(f"Error during training: {e}")
      return float("inf")  # Return high value for failed trials

# Create a Trials object to track hyperparameter evaluations
trials = Trials()

# Run hyperparameter optimization using TPE algorithm
best = fmin(objective, space, algo=tpe.suggest, trials=trials, max_evals=2)




==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
  0%|          | 0/2 [00:00<?, ?trial/s, best loss=?]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.2641595209449405.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Error during training: 'warmup_steps'
==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
 50%|█████     | 1/2 [00:14<00:14, 14.65s/trial, best loss: inf]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.4265864260539507.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Error during training: 'warmup_steps'
100%|██████████| 2/2 [00:28<00:00, 14.27s/trial, best loss: inf]


In [21]:
# Print the best hyperparameters found during optimization
print("Best Hyperparameters:", best)

Best Hyperparameters: {'gradient_accumulation_steps': np.float64(7.0), 'learning_rate': np.float64(0.07486553705955584), 'lora_alpha': np.float64(23.0), 'lora_dropout': np.float64(0.2641595209449405), 'per_device_train_batch_size': np.float64(13.0)}


# Hyperparameter search
**Step 1: Define the Hyperparameter Search Space**
We need to define the search space for the hyperparameters we want to tune. For example, let's say we want to tune the following hyperparameters:

* `learning_rate`
* `per_device_train_batch_size`
* `gradient_accumulation_steps`
* `warmup_steps`
* `num_train_epochs`
* `lora_alpha`
* `lora_dropout`

We can define the search space as follows:

In [22]:
import hyperopt
from hyperopt import hp
from hyperopt import Trials
from hyperopt import fmin, tpe, Trials
# Define the search space for hyperparameters
space = {
  'learning_rate': hp.loguniform('learning_rate', -5, -1),  # Learning rate in log scale
  'lora_alpha': hp.quniform('lora_alpha', 1, 32, 1),  # LoRA alpha with quantized steps
  'lora_dropout': hp.uniform('lora_dropout', 0, 0.5),  # LoRA dropout rate
  # Uncomment these if you want to tune them
  # 'per_device_train_batch_size': hp.quniform('per_device_train_batch_size', 2, 16, 1),
  # 'gradient_accumulation_steps': hp.quniform('gradient_accumulation_steps', 1, 8, 1),
  # 'warmup_steps': hp.quniform('warmup_steps', 0, 1000, 1),
  # 'num_train_epochs': hp.quniform('num_train_epochs', 1, 5, 1),
}

**Step 2. Define the Objective Function**

The objective function is a function that takes in the hyperparameters, sets them in the `config` dictionary, trains the model, and returns the loss or metric to minimize. We need to modify the previous fine-tuning code to define the objective function.

In [23]:
def objective(params):
  # Set hyperparameters in the config dictionary (assuming it's defined elsewhere)
  config['training_config']['learning_rate'] = params['learning_rate']
  config['lora_config']['lora_alpha'] = params['lora_alpha']
  config['lora_config']['lora_dropout'] = params['lora_dropout']
  # ... Set other hyperparameters from params dictionary ...
  #config['training_config']['per_device_train_batch_size'] = params['per_device_train_batch_size']
  #config['training_config']['gradient_accumulation_steps'] = params['gradient_accumulation_steps']
  #config['training_config']['warmup_steps'] = params['warmup_steps']
  #config['training_config']['num_train_epochs'] = params['num_train_epochs']

  # Load the model and tokenizer (assuming these are defined elsewhere)
  try:
      model, tokenizer = FastLanguageModel.from_pretrained(
          model_name=config.get("model_config").get("base_model"),
          max_seq_length=config.get("model_config").get("max_seq_length"),
          dtype=config.get("model_config").get("dtype"),
          load_in_4bit=config.get("model_config").get("load_in_4bit"),
      )
  except Exception as e:
      print(f"Error loading model and tokenizer: {e}")
      return float("inf")  # Return high value for errors

  # Setup LoRA for the model (assuming FastLanguageModel supports LoRA)
  try:
      model = FastLanguageModel.get_peft_model(
          model,
          r=config.get("lora_config").get("r"),
          target_modules=config.get("lora_config").get("target_modules"),
          lora_alpha=params['lora_alpha'],
          lora_dropout=params['lora_dropout'],
          bias=config.get("lora_config").get("bias"),
          use_gradient_checkpointing=config.get("lora_config").get("use_gradient_checkpointing"),
          random_state=42,
          use_rslora=config.get("lora_config").get("use_rslora"),
          use_dora=config.get("lora_config").get("use_dora"),
          loftq_config=config.get("lora_config").get("loftq_config")
      )
  except Exception as e:
      print(f"Error setting up LoRA: {e}")
      return float("inf")  # Return high value for errors
  # Train the model on the test dataset (assuming SFTTrainer and training arguments are defined)
  try:
      trainer = SFTTrainer(
          model=model,
          tokenizer=tokenizer,
          train_dataset=test_dataset,
          dataset_text_field=config_dataset.get("training_dataset").get("input_field"),
          max_seq_length=config.get("model_config").get("max_seq_length"),
          dataset_num_proc=2,
          packing=False,
          args=TrainingArguments(
              per_device_train_batch_size=int(params['per_device_train_batch_size']),
              gradient_accumulation_steps=params['gradient_accumulation_steps'],
              warmup_steps=params['warmup_steps'],
              max_steps=config.get("training_config").get("max_steps"),
              num_train_epochs=params['num_train_epochs'],
              learning_rate=params['learning_rate'],
              fp16=config.get("training_config").get("fp16"),
              bf16=config.get("training_config").get("bf16"),
              logging_steps=config.get("training_config").get("logging_steps"),
              optim=config.get("training_config").get("optim"),
              weight_decay=config.get("training_config").get("weight_decay"),
              lr_scheduler_type=config.get("training_config").get("lr_scheduler_type"),
              seed=42,
              output_dir=config.get("training_config").get("output_dir")
          )
      )
      trainer_stats = trainer.train()
      return trainer_stats.loss  # Assuming loss is the metric to minimize
  except Exception as e:
      print(f"Error during training: {e}")
      return float("inf")  # Return high value for failed trials



**Step 3: Perform Hyperparameter Search**

Now that we have defined the objective function, we can perform the hyperparameter search using Hyperopt's `fmin` function. We need to specify the objective function, the search space, and the maximum number of evaluations.

In [24]:

# Create a Trials object to track hyperparameter evaluations
trials = Trials()
# Run hyperparameter optimization using TPE algorithm
best = fmin(objective, space, algo=tpe.suggest, trials=trials, max_evals=2)
# Print the best hyperparameters found during optimization
print("Best Hyperparameters:", best)

==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
  0%|          | 0/2 [00:00<?, ?trial/s, best loss=?]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.32416532715212426.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Error during training: 'per_device_train_batch_size'
==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
 50%|█████     | 1/2 [00:14<00:14, 14.30s/trial, best loss: inf]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.28715709652682314.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Error during training: 'per_device_train_batch_size'
100%|██████████| 2/2 [00:28<00:00, 14.22s/trial, best loss: inf]
Best Hyperparameters: {'learning_rate': np.float64(0.03322899701821198), 'lora_alpha': np.float64(17.0), 'lora_dropout': np.float64(0.32416532715212426)}


In [25]:
from huggingface_hub import login, logout

In [26]:
#login(token) # non-blocking login

from huggingface_hub import login
login("YOUR_HF_TOKEN_HERE")

In [27]:
import torch
import gc
def reset_gpu_memory():
    torch.cuda.empty_cache()
    gc.collect()
    print("GPU memory cleared!")
# Example usage:
reset_gpu_memory()

GPU memory cleared!


Best Hyperparameters: {'learning_rate': 0.03347123299210303, 'lora_alpha': 19.0, 'lora_dropout': 0.4819141472093197}

Best Hyperparameters: {'gradient_accumulation_steps': 8.0, 'learning_rate': 0.23274337759179295, 'lora_alpha': 8.0, 'lora_dropout': 0.0491660925212421, 'per_device_train_batch_size': 13.0}

Best Hyperparameters: {'gradient_accumulation_steps': 4.0, 'learning_rate': 0.186066529001672, 'lora_alpha': 32.0, 'lora_dropout': 0.24368804023352264, 'per_device_train_batch_size': 10.0}

Best Hyperparameters: {'learning_rate': 0.011846192509972951, 'lora_alpha': 8.0, 'lora_dropout': 0.2087248476879589}



Best Hyperparameters (Likely Fastest): {'learning_rate': 1.881999040862022e-05, 'per_device_train_batch_size': 2, 'gradient_accumulation_steps': 2, 'lora_alpha': 29}
Best Training Time: 48.178661584854126 seconds


In [34]:
# Defining the configuration for the base model, LoRA and training
config = {
    "hugging_face_username":"ruslanmv",
    "model_config": {
        "base_model":"mistralai/Mistral-7B-Instruct-v0.2", # The base model
        "finetuned_model":"aravag/Mercury-STD-Mistral", # The finetuned model
        "max_seq_length": 2048, # The maximum sequence length
       # "dtype":torch.float16, # The data type
       #  "dtype": torch.float32,  # Use float32 instead of half CUDA capability < 8
          "dtype" : None, # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+

        "load_in_4bit": True, # Load the model in 4-bit
    },
    "lora_config": {
      "r": 16, # The number of LoRA layers 8, 16, 32, 64
      "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # The target modules
      #"lora_alpha":16, # The alpha value for LoRA
      "lora_alpha":29, # The alpha value for LoRA by search grid
      "lora_dropout":0, # The dropout value for LoRA
      "bias":"none", # The bias for LoRA
      "use_gradient_checkpointing":True, # Use gradient checkpointing
      "use_rslora":False, # Use RSLora
      "use_dora":False, # Use DoRa
      "loftq_config":None # The LoFTQ configuration
    },

    "training_config": {
        #"per_device_train_batch_size": 2, # The batch size
        "per_device_train_batch_size": 2, # The batch size by search grid
        #"gradient_accumulation_steps": 4, # The gradient accumulation steps
        "gradient_accumulation_steps": 2, # The gradient accumulation steps by search grid
        "warmup_steps": 5, # The warmup steps
        "max_steps":0, # The maximum steps (0 if the epochs are defined)
        "num_train_epochs": 1, # The number of training epochs(0 if the maximum steps are defined)
        #"learning_rate": 2e-4, # The learning rate
        "learning_rate": 1.88e-05, # The learning rate  by search grid
        "fp16": not torch.cuda.is_bf16_supported(), # The fp16
        "bf16": torch.cuda.is_bf16_supported(), # The bf16
        "logging_steps": 1, # The logging steps
        "optim" :"adamw_8bit", # The optimizer
        "weight_decay" : 0.01,  # The weight decay
        "lr_scheduler_type": "linear", # The learning rate scheduler
        "seed" : 42, # The seed
        "output_dir" : "outputs", # The output directory
    }
}

In [35]:
# Loading the model and the tokinizer for the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = config.get("model_config").get("base_model"),
    max_seq_length = config.get("model_config").get("max_seq_length"),
    dtype = config.get("model_config").get("dtype"),
    load_in_4bit = config.get("model_config").get("load_in_4bit"),

)

==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [36]:
# Set up GPU acceleration
if torch.cuda.device_count() > 1:
    print("Multiple GPUs enabled")
    devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())]
    model_parallel = torch.nn.DataParallel(model, device_ids=[0, 1])
    # Access the original model from the DataParallel object
    model = model_parallel.module
else:
    print("No DataParallel ")
    #device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


No DataParallel 


In [ ]:
#model = model.half()  # the model to half precision (float16)

In [37]:
# Setup for QLoRA/LoRA peft of the base model
model = FastLanguageModel.get_peft_model(
    model,
    r = config.get("lora_config").get("r"),
    target_modules = config.get("lora_config").get("target_modules"),
    lora_alpha = config.get("lora_config").get("lora_alpha"),
    lora_dropout = config.get("lora_config").get("lora_dropout"),
    bias = config.get("lora_config").get("bias"),
    use_gradient_checkpointing = config.get("lora_config").get("use_gradient_checkpointing"),
    random_state = 42,
    use_rslora = config.get("lora_config").get("use_rslora"),
    use_dora = config.get("lora_config").get("use_dora"),
    loftq_config = config.get("lora_config").get("loftq_config"),
)


In [38]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.get("model_config").get("base_model"))

In [39]:
tokenizer.add_eos_token = True
tokenizer.pad_token_id = 0
tokenizer.padding_side = "left"

In [40]:
config_dataset={    "training_dataset": {
        "name": "ruslanmv/ai-medical-dataset", # The dataset name(huggingface/datasets)
        "split": "train",  # The dataset split
        "input_fields": ["question", "context"] ,# The input fields
        "input_field": "text",# The input field
    },
                }

In [41]:
config_dataset.get("training_dataset")

{'name': 'ruslanmv/ai-medical-dataset',
 'split': 'train',
 'input_fields': ['question', 'context'],
 'input_field': 'text'}

In [42]:
# Loading the training dataset
train_dataset = load_dataset(config_dataset.get("training_dataset").get("name"), split = config_dataset.get("training_dataset").get("split"))

README.md:   0%|          | 0.00/2.97k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

data/train-00000-of-00018.parquet:   0%|          | 0.00/556M [00:00<?, ?B/s]

data/train-00001-of-00018.parquet:   0%|          | 0.00/542M [00:00<?, ?B/s]

...

data/train-00015-of-00018.parquet:   0%|          | 0.00/71.2M [00:00<?, ?B/s]

data/train-00016-of-00018.parquet:   0%|          | 0.00/71.2M [00:00<?, ?B/s]

data/train-00017-of-00018.parquet:   0%|          | 0.00/71.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21210000 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

In [43]:
train_dataset

Dataset({
    features: ['question', 'context'],
    num_rows: 21210000
})

In [44]:
# Select the first 10 rows of the dataset
test_dataset = train_dataset.select(range(100))

In [45]:
test_dataset

Dataset({
    features: ['question', 'context'],
    num_rows: 100
})

In [46]:
test_dataset[1]

{'question': 'What isoform is a locus for inherited spinocerebellar at',
 'context': "Rapid firing of cerebellar Purkinje neurons is facilitated in part by a voltage-gated Na+ (NaV) 'resurgent' current, which allows renewed Na+ influx during membrane repolarization. Resurgent current results from unbinding of a blocking particle that competes with normal channel inactivation. The underlying molecular components contributing to resurgent current have not been fully identified. Here, we show that the NaV channel auxiliary subunit FGF14 'b' isoform, a locus for inherited spinocerebellar ataxias, controls resurgent current and repetitive firing in Purkinje neurons. FGF14 knockdown biased NaV channels towards the inactivated state by decreasing channel availability, diminishing the 'late' NaV current, and accelerating channel inactivation rate, thereby reducing resurgent current and repetitive spiking. Critical for these effects was both the alternatively spliced FGF14b N-terminus and direc

In [47]:
medical_prompt = """You are an AI Medical Assistant Chatbot, trained to answer medical questions. Below is an instruction that describes a task, paired with an response context. Write a response that appropriately completes the request.

### Instruction:
{}


### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["question"]
    outputs      = examples["context"]
    texts = []
    for instruction, output in zip(instructions,  outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = medical_prompt.format(instruction,  output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

In [48]:
test_dataset= test_dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [49]:
test_dataset

Dataset({
    features: ['question', 'context', 'text'],
    num_rows: 100
})

In [50]:
test_dataset['text'][1]

"You are an AI Medical Assistant Chatbot, trained to answer medical questions. Below is an instruction that describes a task, paired with an response context. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat isoform is a locus for inherited spinocerebellar at\n\n\n### Response:\nRapid firing of cerebellar Purkinje neurons is facilitated in part by a voltage-gated Na+ (NaV) 'resurgent' current, which allows renewed Na+ influx during membrane repolarization. Resurgent current results from unbinding of a blocking particle that competes with normal channel inactivation. The underlying molecular components contributing to resurgent current have not been fully identified. Here, we show that the NaV channel auxiliary subunit FGF14 'b' isoform, a locus for inherited spinocerebellar ataxias, controls resurgent current and repetitive firing in Purkinje neurons. FGF14 knockdown biased NaV channels towards the inactivated state by decreasing channel availability

In [51]:
is_test=False #to train on the entire dataset
if is_test:
    train_dataset=test_dataset
else:
    train_dataset= train_dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/21210000 [00:00<?, ? examples/s]

In [52]:
train_dataset['text'][1]

"You are an AI Medical Assistant Chatbot, trained to answer medical questions. Below is an instruction that describes a task, paired with an response context. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat isoform is a locus for inherited spinocerebellar at\n\n\n### Response:\nRapid firing of cerebellar Purkinje neurons is facilitated in part by a voltage-gated Na+ (NaV) 'resurgent' current, which allows renewed Na+ influx during membrane repolarization. Resurgent current results from unbinding of a blocking particle that competes with normal channel inactivation. The underlying molecular components contributing to resurgent current have not been fully identified. Here, we show that the NaV channel auxiliary subunit FGF14 'b' isoform, a locus for inherited spinocerebellar ataxias, controls resurgent current and repetitive firing in Purkinje neurons. FGF14 knockdown biased NaV channels towards the inactivated state by decreasing channel availability

In [53]:
# Setting up the trainer for the model
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = config_dataset.get("training_dataset").get("input_field"),
    max_seq_length = config.get("model_config").get("max_seq_length"),
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = config.get("training_config").get("per_device_train_batch_size"),
        gradient_accumulation_steps = config.get("training_config").get("gradient_accumulation_steps"),
        warmup_steps = config.get("training_config").get("warmup_steps"),
        max_steps = config.get("training_config").get("max_steps"),
        num_train_epochs= config.get("training_config").get("num_train_epochs"),
        learning_rate = config.get("training_config").get("learning_rate"),
        fp16 = config.get("training_config").get("fp16"),
        bf16 = config.get("training_config").get("bf16"),
        logging_steps = config.get("training_config").get("logging_steps"),
        optim = config.get("training_config").get("optim"),
        weight_decay = config.get("training_config").get("weight_decay"),
        lr_scheduler_type = config.get("training_config").get("lr_scheduler_type"),
        seed = 42,
        output_dir = config.get("training_config").get("output_dir"),
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/21210000 [00:00<?, ? examples/s]

In [54]:
# Memory statistics before training
gpu_statistics = torch.cuda.get_device_properties(0)
reserved_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
max_memory = round(gpu_statistics.total_memory / 1024**3, 2)
print(f"Reserved Memory: {reserved_memory}GB")
print(f"Max Memory: {max_memory}GB")

Reserved Memory: 13.62GB
Max Memory: 14.56GB


In [ ]:
##  [ 1038/2651250 53:49 < 2295:10:28, 0.32 it/s, Epoch 0.00/1] old

In [55]:
# Training the model
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 21,210,000 | Num Epochs = 1 | Total steps = 0
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss
1,2.930023


In [56]:
# Memory statistics after training
used_memory = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
used_memory_lora = round(used_memory - reserved_memory, 2)
used_memory_persentage = round((used_memory / max_memory) * 100, 2)
used_memory_lora_persentage = round((used_memory_lora / max_memory) * 100, 2)
print(f"Used Memory: {used_memory}GB ({used_memory_persentage}%)")
print(f"Used Memory for training(fine-tuning) LoRA: {used_memory_lora}GB ({used_memory_lora_persentage}%)")

Used Memory: 13.61GB (93.48%)
Used Memory for training(fine-tuning) LoRA: -0.01GB (-0.07%)


In [57]:
# Saving the trainer stats
with open("trainer_stats.json", "w") as f:
    json.dump(trainer_stats, f, indent=4)

In [60]:
# Locally saving the model and pushing it to the Hugging Face Hub (only LoRA adapters)
model.save_pretrained(config.get("model_config").get("finetuned_model"))
model.push_to_hub(config.get("model_config").get("finetuned_model"))
tokenizer.save_pretrained(config.get("model_config").get("finetuned_model"))
tokenizer.push_to_hub(config.get("model_config").get("finetuned_model"))

README.md:   0%|          | 0.00/574 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  47%|####7     | 79.1MB /  168MB            

Saved model to https://huggingface.co/aravag/Mercury-STD-Mistral


CommitInfo(commit_url='https://huggingface.co/aravag/Mercury-STD-Mistral/commit/694a9bd89d5963ddfbed2176283a7f0929af8d4f', commit_message='Upload tokenizer', commit_description='', oid='694a9bd89d5963ddfbed2176283a7f0929af8d4f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aravag/Mercury-STD-Mistral', endpoint='https://huggingface.co', repo_type='model', repo_id='aravag/Mercury-STD-Mistral'), pr_revision=None, pr_num=None)

In [62]:
# SKIP THIS CELL - not enough disk space on Colab free tier


# # Saving the model using merged_16bit(float16), merged_4bit(int4) or quantization options(q8_0, q4_k_m, q5_k_m)...
# model.save_pretrained_merged(config.get("model_config").get("finetuned_model"), tokenizer, save_method = "merged_16bit",)
# model.push_to_hub_merged(config.get("model_config").get("finetuned_model"), tokenizer, save_method = "merged_16bit")

# model.save_pretrained_merged(config.get("model_config").get("finetuned_model"), tokenizer, save_method = "merged_4bit",)
# model.push_to_hub_merged(config.get("model_config").get("finetuned_model"), tokenizer, save_method = "merged_4bit")

# model.save_pretrained_gguf(config.get("model_config").get("finetuned_model"), tokenizer)
# model.push_to_hub_gguf(config.get("model_config").get("finetuned_model"), tokenizer)

# model.save_pretrained_gguf(config.get("model_config").get("finetuned_model"), tokenizer, quantization_method = "f16")
# model.push_to_hub_gguf(config.get("model_config").get("finetuned_model"), tokenizer, quantization_method = "f16")

# model.save_pretrained_gguf(config.get("model_config").get("finetuned_model"), tokenizer, quantization_method = "q4_k_m")
# model.push_to_hub_gguf(config.get("model_config").get("finetuned_model"), tokenizer, quantization_method = "q4_k_m")

In [63]:
# Loading the fine-tuned model and the tokenizer for inference
model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = config.get("model_config").get("finetuned_model"),
        max_seq_length = config.get("model_config").get("max_seq_length"),
        dtype = config.get("model_config").get("dtype"),
        load_in_4bit = config.get("model_config").get("load_in_4bit"),
    )

# Using FastLanguageModel for fast inference
FastLanguageModel.for_inference(model)

# Tokenizing the input and generating the output
inputs = tokenizer(
[
    "<|start_header_id|>system<|end_header_id|> You are a Medical AI chatbot assistant .<|eot_id|><|start_header_id|>user<|end_header_id|> This is the question: What was the main cause of the inflammatory CD4+ T cells?<|eot_id|>"
], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
tokenizer.batch_decode(outputs, skip_special_tokens = True)

==((====))==  Unsloth 2026.5.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


["<|start_header_id|>system<|end_header_id|> You are a Medical AI chatbot assistant .<|eot_id|><|start_header_id|>user<|end_header_id|> This is the question: What was the main cause of the inflammatory CD4+ T cells?<|eot_id|> I'm here to help answer any medical-related questions you might have. However, I'll need some more context to provide an accurate answer to your question. Inflammatory CD4+ T cells can be caused by a variety of conditions, including infections, autoimmune diseases, and certain medications. Without more information about the specific situation you're asking about, it's difficult to pinpoint the exact cause. If you could provide some additional details, such as any symptoms you're experiencing, any medications you're taking, or any known health conditions you have, I'd be happy to try and help you further."]